In [ ]:

# ============================================================
# AUTOMATED SOFTWARE TESTING WITH APACHE JMETER
# Google Colab - Single Cell Workshop
#
# Functional Testing + Non-Functional Testing
# ============================================================

import os
import time
import shutil
import subprocess
import requests
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from IPython.display import display, HTML
from google.colab import files

print("=" * 60)
print("AUTOMATED SOFTWARE TESTING WITH APACHE JMETER")
print("=" * 60)


# ============================================================
# STEP 1: INSTALL JAVA AND APACHE JMETER
# ============================================================

print("\n[1/9] Installing Java and Apache JMeter...")

subprocess.run(
    "apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless wget",
    shell=True,
    check=True
)

if not os.path.exists("/content/apache-jmeter-5.6.3"):
    subprocess.run(
        "wget -q https://archive.apache.org/dist/jmeter/binaries/apache-jmeter-5.6.3.tgz",
        shell=True,
        check=True
    )

    subprocess.run(
        "tar -xzf apache-jmeter-5.6.3.tgz",
        shell=True,
        check=True
    )

JMETER = "/content/apache-jmeter-5.6.3/bin/jmeter"

subprocess.run([JMETER, "--version"], check=True)

subprocess.run(
    ["pip", "install", "-q", "flask", "requests", "pandas", "matplotlib"],
    check=True
)

print("Java and JMeter installed!")


# ============================================================
# STEP 2: CREATE SIMPLE BANKING APPLICATION
# ============================================================

print("\n[2/9] Creating banking API...")

app_code = '''
from flask import Flask, jsonify, request

app = Flask(__name__)

@app.route("/health")
def health():
    return jsonify(status="OK")

@app.route("/balance")
def balance():
    return jsonify(
        account="1234567890",
        balance=1000000,
        currency="IDR"
    )

@app.route("/transfer", methods=["POST"])
def transfer():
    data = request.get_json(silent=True) or {}

    return jsonify(
        status="SUCCESS",
        amount=data.get("amount", 0)
    )

if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False
    )
'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

# Start Flask API
app_process = subprocess.Popen(
    ["python", "/content/app.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

BASE_URL = "http://127.0.0.1:5000"

# Wait for application
for attempt in range(30):
    try:
        response = requests.get(BASE_URL + "/health")
        if response.status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    raise RuntimeError("Banking API failed to start")

print("Banking API started successfully!")


# ============================================================
# STEP 3: VERIFY BANKING APPLICATION
# ============================================================

print("\n[3/9] Verifying banking API...")

print("Health:", requests.get(
    BASE_URL + "/health"
).json())

print("Balance:", requests.get(
    BASE_URL + "/balance"
).json())

print("Transfer:", requests.post(
    BASE_URL + "/transfer",
    json={"amount": 100000}
).json())

print("API verification completed!")


# ============================================================
# STEP 4: CREATE JMETER TEST PLAN
# ============================================================

print("\n[4/9] Creating JMeter test plan...")

def create_test_plan(filename, users=1, loops=1, ramp=1):

    root = ET.Element("jmeterTestPlan", {
        "version": "1.2",
        "properties": "5.0",
        "jmeter": "5.0"
    })

    main_tree = ET.SubElement(root, "hashTree")

    plan = ET.SubElement(main_tree, "TestPlan", {
        "guiclass": "TestPlanGui",
        "testclass": "TestPlan",
        "testname": "Banking API Test",
        "enabled": "true"
    })

    ET.SubElement(plan, "stringProp", {
        "name": "TestPlan.comments"
    }).text = "Automated Banking API Testing"

    ET.SubElement(plan, "boolProp", {
        "name": "TestPlan.functional_mode"
    }).text = "false"

    ET.SubElement(plan, "boolProp", {
        "name": "TestPlan.serialize_threadgroups"
    }).text = "false"

    variables = ET.SubElement(plan, "elementProp", {
        "name": "TestPlan.user_defined_variables",
        "elementType": "Arguments"
    })

    ET.SubElement(variables, "collectionProp", {
        "name": "Arguments.arguments"
    })

    plan_hash = ET.SubElement(main_tree, "hashTree")

    group = ET.SubElement(plan_hash, "ThreadGroup", {
        "guiclass": "ThreadGroupGui",
        "testclass": "ThreadGroup",
        "testname": "Banking API Users",
        "enabled": "true"
    })

    for key, value in {
        "ThreadGroup.num_threads": users,
        "ThreadGroup.ramp_time": ramp,
        "ThreadGroup.on_sample_error": "continue"
    }.items():

        ET.SubElement(group, "stringProp", {
            "name": key
        }).text = str(value)

    ET.SubElement(group, "boolProp", {
        "name": "ThreadGroup.same_user_on_next_iteration"
    }).text = "true"

    controller = ET.SubElement(group, "elementProp", {
        "name": "ThreadGroup.main_controller",
        "elementType": "LoopController",
        "guiclass": "LoopControlPanel",
        "testclass": "LoopController",
        "testname": "Loop Controller",
        "enabled": "true"
    })

    ET.SubElement(controller, "boolProp", {
        "name": "LoopController.continue_forever"
    }).text = "false"

    ET.SubElement(controller, "stringProp", {
        "name": "LoopController.loops"
    }).text = str(loops)

    group_hash = ET.SubElement(plan_hash, "hashTree")

    endpoints = [
        ("TC01 Health Check", "/health", "GET", None, '"OK"'),
        ("TC02 Account Balance", "/balance", "GET", None, "1000000"),
        ("TC03 Money Transfer", "/transfer", "POST",
         '{"amount":100000}', '"SUCCESS"')
    ]

    for name, path, method, body, expected in endpoints:

        sampler = ET.SubElement(group_hash, "HTTPSamplerProxy", {
            "guiclass": "HttpTestSampleGui",
            "testclass": "HTTPSamplerProxy",
            "testname": name,
            "enabled": "true"
        })

        arguments = ET.SubElement(sampler, "elementProp", {
            "name": "HTTPsampler.Arguments",
            "elementType": "Arguments"
        })

        argument_list = ET.SubElement(arguments, "collectionProp", {
            "name": "Arguments.arguments"
        })

        if body:

            argument = ET.SubElement(argument_list, "elementProp", {
                "name": "",
                "elementType": "HTTPArgument"
            })

            ET.SubElement(argument, "boolProp", {
                "name": "HTTPArgument.always_encode"
            }).text = "false"

            ET.SubElement(argument, "stringProp", {
                "name": "Argument.value"
            }).text = body

            ET.SubElement(argument, "stringProp", {
                "name": "Argument.metadata"
            }).text = "="

            ET.SubElement(argument, "boolProp", {
                "name": "HTTPArgument.use_equals"
            }).text = "true"

        properties = {
            "HTTPSampler.domain": "127.0.0.1",
            "HTTPSampler.port": "5000",
            "HTTPSampler.protocol": "http",
            "HTTPSampler.path": path,
            "HTTPSampler.method": method,
            "HTTPSampler.contentEncoding": "UTF-8",
            "HTTPSampler.connect_timeout": "5000",
            "HTTPSampler.response_timeout": "5000",
            "HTTPSampler.implementation": "HttpClient4"
        }

        for key, value in properties.items():

            ET.SubElement(sampler, "stringProp", {
                "name": key
            }).text = value

        ET.SubElement(sampler, "boolProp", {
            "name": "HTTPSampler.postBodyRaw"
        }).text = str(bool(body)).lower()

        sampler_hash = ET.SubElement(group_hash, "hashTree")

        if body:

            headers = ET.SubElement(sampler_hash, "HeaderManager", {
                "guiclass": "HeaderPanel",
                "testclass": "HeaderManager",
                "testname": "HTTP Headers",
                "enabled": "true"
            })

            header_list = ET.SubElement(headers, "collectionProp", {
                "name": "HeaderManager.headers"
            })

            header = ET.SubElement(header_list, "elementProp", {
                "name": "",
                "elementType": "Header"
            })

            ET.SubElement(header, "stringProp", {
                "name": "Header.name"
            }).text = "Content-Type"

            ET.SubElement(header, "stringProp", {
                "name": "Header.value"
            }).text = "application/json"

            ET.SubElement(sampler_hash, "hashTree")

        assertion = ET.SubElement(sampler_hash, "ResponseAssertion", {
            "guiclass": "AssertionGui",
            "testclass": "ResponseAssertion",
            "testname": "Validate Response",
            "enabled": "true"
        })

        strings = ET.SubElement(assertion, "collectionProp", {
            "name": "Asserion.test_strings"
        })

        ET.SubElement(strings, "stringProp", {
            "name": ""
        }).text = expected

        ET.SubElement(assertion, "stringProp", {
            "name": "Assertion.custom_message"
        }).text = "Unexpected API response"

        ET.SubElement(assertion, "stringProp", {
            "name": "Assertion.test_field"
        }).text = "Assertion.response_data"

        ET.SubElement(assertion, "boolProp", {
            "name": "Assertion.assume_success"
        }).text = "false"

        ET.SubElement(assertion, "intProp", {
            "name": "Assertion.test_type"
        }).text = "2"

        ET.SubElement(sampler_hash, "hashTree")

    ET.indent(root, space="  ")

    ET.ElementTree(root).write(
        filename,
        encoding="UTF-8",
        xml_declaration=True
    )

    print("Created:", filename)


os.makedirs("/content/results", exist_ok=True)

# Generate functional test plan
create_test_plan("/content/banking.jmx")


# ============================================================
# STEP 5: FUNCTIONAL TESTING
# ============================================================

print("\n[5/9] Running functional testing...")

result = subprocess.run([
    JMETER,
    "-n",
    "-t", "/content/banking.jmx",
    "-l", "/content/results/functional.csv"
], capture_output=True, text=True)

print(result.stdout[-2000:])

if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Functional testing failed")

print("Functional testing completed!")


# ============================================================
# STEP 6: NON-FUNCTIONAL / PERFORMANCE TESTING
# ============================================================

print("\n[6/9] Running performance testing...")

# 10 users x 10 iterations x 3 APIs = 300 requests
create_test_plan(
    "/content/performance.jmx",
    users=10,
    loops=10,
    ramp=5
)

report_path = "/content/results/html-report"

if os.path.exists(report_path):
    shutil.rmtree(report_path)

result = subprocess.run([
    JMETER,
    "-n",
    "-t", "/content/performance.jmx",
    "-l", "/content/results/performance.csv",
    "-e",
    "-o", report_path
], capture_output=True, text=True)

print(result.stdout[-2000:])

if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("Performance testing failed")

print("Performance testing completed!")


# ============================================================
# STEP 7: ANALYZE PERFORMANCE RESULTS
# ============================================================

print("\n[7/9] Analyzing test results...")

df = pd.read_csv("/content/results/performance.csv")

total = len(df)

success = (
    df["success"].astype(str).str.lower() == "true"
)

failed = total - success.sum()

duration = (
    (df["timeStamp"] + df["elapsed"]).max()
    - df["timeStamp"].min()
) / 1000

throughput = total / duration if duration > 0 else 0

print("\n===== JMETER PERFORMANCE RESULTS =====")

print("Total Requests:", total)
print("Successful Requests:", success.sum())
print("Failed Requests:", failed)

print("Average Response Time:",
      round(df["elapsed"].mean(), 2), "ms")

print("Minimum Response Time:",
      df["elapsed"].min(), "ms")

print("Maximum Response Time:",
      df["elapsed"].max(), "ms")

print("95th Percentile:",
      round(df["elapsed"].quantile(0.95), 2), "ms")

print("Throughput:",
      round(throughput, 2), "requests/second")

print("Error Rate:",
      round(failed / total * 100, 2), "%")

print("\n===== ENDPOINT SUMMARY =====")

summary = df.groupby("label").agg(
    requests=("label", "count"),
    average_response_ms=("elapsed", "mean"),
    errors=("success", lambda x:
        (x.astype(str).str.lower() != "true").sum())
)

display(summary)


# ============================================================
# STEP 8: VISUALIZE PERFORMANCE
# ============================================================

print("\n[8/9] Generating performance chart...")

plt.figure(figsize=(10, 4))

plt.hist(df["elapsed"], bins=20)

plt.title("Banking API Response Time")
plt.xlabel("Response Time (milliseconds)")
plt.ylabel("Number of Requests")

plt.show()


# ============================================================
# STEP 9: DOWNLOAD JMETER HTML REPORT
# ============================================================

print("\n[9/9] Preparing HTML performance report...")

shutil.make_archive(
    "/content/jmeter-report",
    "zip",
    report_path
)

print("\n" + "=" * 60)
print("WORKSHOP COMPLETED SUCCESSFULLY!")
print("=" * 60)

print("Functional test results:")
print("/content/results/functional.csv")

print("\nPerformance test results:")
print("/content/results/performance.csv")

print("\nHTML report:")
print("/content/jmeter-report.zip")

files.download("/content/jmeter-report.zip")